# Multivariate OLS

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import scipy as sp
import scipy.stats as stats

import pyreadr 
import matplotlib.font_manager as fm
from matplotlib import ticker


# Computing Corner

**1.**

Same as before we just add the additional variables to the `formula=` parameter of our `smf.ols` and for heteroscedasticity-consistent standard errors we will use `fit(cov_type="HC1")`

**2.**

To assess multicollinearity we have to find $R^2_j$ for each variable, For example, we can do so by: 

In [14]:
# We make a random dataframe

df1= pd.DataFrame({"Y": np.random.standard_normal(10),
                   "X1": np.arange(0,10),
                   "X2": np.random.uniform(4,6,10),
                   "X3": np.random.uniform(0,1,10)})

display(df1.head())

,Y,X1,X2,X3
0,-0.388995,0,5.006128,0.206517
1,-1.513089,1,4.062544,0.871560
2,-1.033232,2,5.898303,0.771403
3,-0.039824,3,5.250782,0.051662
4,-0.129431,4,4.680570,0.956368


In [15]:
# Auxiliary Regression between each variable

AuxReg1= smf.ols(
    formula="X1 ~ X2 + X3",
    data= df1
).fit()

print(AuxReg1.rsquared)

AuxReg2= smf.ols(
    formula="X2 ~ X1 + X3",
    data= df1
).fit()

print(AuxReg2.rsquared)

AuxReg3= smf.ols(
    formula="X3 ~ X2 + X1",
    data= df1
).fit()

print(AuxReg3.rsquared)

0.10995948668384159
0.10849561713648592
0.029626377207914945


We can now also calculate VIF as follows

In [26]:
vif_X1 = 1 / (1 - AuxReg1.rsquared)
vif_X2 = 1 / (1 - AuxReg2.rsquared)
vif_X3 = 1 / (1 - AuxReg3.rsquared)

vif= [vif_X1, vif_X2, vif_X3]

for i in vif:
    print("\nVIF X" + str(vif.index(i)+1)+":" , f"{i:.4f}")
    if i < 2:
        print("Under 2, No Serious Multicollinearity")
    elif i > 2 and i <5:
        print("Less than 5, Moderate Multicollinearity")
    if i > 5:
        print("More than 5, Serious Multicollinearity")


VIF X1: 1.1235
Under 2, No Serious Multicollinearity

VIF X2: 1.1217
Under 2, No Serious Multicollinearity

VIF X3: 1.0305
Under 2, No Serious Multicollinearity


**3.**

Unfortunately, statsmodels does not have an automatic scaling function, what we can do however is use either zscore from scipy for automatic opaque procedure or we can go for transparent manual transformation

In [32]:
df1_std_auto= df1.copy()

df1_std_auto[["Y", "X1", "X2", "X3"]] = df1_std_auto[["Y", "X1", "X2", "X3"]].apply(stats.zscore)

display(df1_std_auto.head())

,Y,X1,X2,X3
0,0.065117,-1.566699,-0.131961,-0.837092
1,-1.040248,-1.218544,-2.002998,1.105562
2,-0.568386,-0.870388,1.637137,0.812995
3,0.408470,-0.522233,0.353165,-1.289439
4,0.320356,-0.174078,-0.777511,1.353294


In [31]:
df_std_pd = (df1 - df1.mean()) / df1.std()

display(df_std_pd.head())

,Y,X1,X2,X3
0,0.061775,-1.486301,-0.125189,-0.794135
1,-0.986866,-1.156012,-1.900210,1.048828
2,-0.539218,-0.825723,1.553125,0.771275
3,0.387509,-0.495434,0.335041,-1.223270
4,0.303917,-0.165145,-0.737612,1.283848


The difference in values is due to the way Scipy works, it actually uses population standard deviation i.e. it used the formula

$$
\sigma= \sqrt{\frac{\sum (x_i - \bar{x})^2}{n} }
$$

While pandas uses the formula for sample standard deviation

$$
\sigma= \sqrt{\frac{\sum (x_i - \bar{x})^2}{n-1} }
$$

In [35]:
# To make them same

df1_std_auto= df1.copy()

df1_std_auto[["Y", "X1", "X2", "X3"]] = df1_std_auto[["Y", "X1", "X2", "X3"]].apply(stats.zscore, ddof=1)

display(df1_std_auto.head())

,Y,X1,X2,X3
0,0.061775,-1.486301,-0.125189,-0.794135
1,-0.986866,-1.156012,-1.900210,1.048828
2,-0.539218,-0.825723,1.553125,0.771275
3,0.387509,-0.495434,0.335041,-1.223270
4,0.303917,-0.165145,-0.737612,1.283848


In [ ]:
#Checking if both are roughly equal
np.allclose(df1_std_auto, df_std_pd)

True

It's better to keep ddof explicit even if we need population standard deviation, so that it is not missed. Most of the time sample population standard deviation will be used

**4.**

F-test can be done manually as follows:



In [43]:
# We fit the unrestricted model

unrestricted = smf.ols(
    formula= "Y~ X1+X2+X3",
    data= df1
).fit()

In [ ]:
# Now we fit the restricted models

# H0: β1=β2=0
restricted_the= smf.ols(          # We call this F test as "THE" F test, i.e the one in which all βs are 0
    formula="Y~X3",
    data= df1
).fit()

# H0: β1=β2
df1["X1X2"]= df1["X1"]+df1["X2"]

restricted = smf.ols(
    "Y ~ X1X2 + X3",
    data=df1
).fit()


In [57]:
# The F-Statistic for # H0: β1=β2=0

Ru = unrestricted.rsquared
Rr = restricted_the.rsquared

q = 2
df = unrestricted.df_resid    # n-k-1

# F statistic
F1 = ((Ru - Rr)/q) / ((1 - Ru)/df)

print("F statistic:", F1)

# p-value
p1 = 1 - stats.f.cdf(F1, dfn=q, dfd=df)

print("p-value:", p1)

F statistic: 0.14151672652873257
p-value: 0.87085456948311


In [ ]:
# F-Statistic for # H0: β1=β2

Ru= unrestricted.rsquared
Rr= restricted.rsquared

q=1
df= unrestricted.df_resid

# F statistic
F2 = ((Ru - Rr)/q) / ((1 - Ru)/df)

print("F statistic:", F2)

# p-value
p2 = 1 - stats.f.cdf(F2, dfn=q, dfd=df) #dof in numerator and dof in denominator

print("p-value:", p2)

F statistic: 0.24376586012779317
p-value: 0.639050083848141


This is something that can be done to have more control over the process and understanding what the F test actually does, however in most practical cases we will be using the automatic way.

F test can be done automatically using:

In [53]:
print(unrestricted.f_test("X1 = 0, X2 = 0"))
print(unrestricted.f_test("X1 = X2"))

<F test: F=0.14151672652873318, p=0.8708545694831095, df_denom=6, df_num=2>
<F test: F=0.24376586012779364, p=0.6390500838481408, df_denom=6, df_num=1>


In [61]:
# A quick check

#Automatic Ftest
the_f_test= unrestricted.f_test("X1=0,X2=0")
f_test= unrestricted.f_test("X1 = X2")
# Their values can be accessed using .pvalue and .favalue


print(np.allclose(the_f_test.pvalue, p1))
print(np.allclose(the_f_test.fvalue, F1))

print(np.allclose(f_test.pvalue, p2))
print(np.allclose(f_test.fvalue, F2))

True
True
True
True


**5.**

In [62]:
unrestricted.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      Y   R-squared:                       0.045
Model:                            OLS   Adj. R-squared:                 -0.432
Method:                 Least Squares   F-statistic:                   0.09465
Date:                Fri, 10 Jul 2026   Prob (F-statistic):              0.960
Time:                        16:29:51   Log-Likelihood:                -14.126
No. Observations:                  10   AIC:                             36.25
Df Residuals:                       6   BIC:                             37.46
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.8142      4.298      0.422      0.688      -8.702      12.331
X1             0.0066      0.150      0.044      0.966      -0.360       0.373
X2            -0.4419      0.852     -0.519      0.623      -2.527       1.643
X3            -0.1167      1.203     -0.097      0.926      -3.060       2.827
==============================================================================
Omnibus:                        0.617   Durbin-Watson:                   1.781
Prob(Omnibus):                  0.735   Jarque-Bera (JB):                0.330
Skew:                          -0.391   Prob(JB):                        0.848
Kurtosis:                       2.575   Cond. No.                         77.7
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

The F-statistic and F-test reported is for THE F-test, i.e 
$$
H_0: \beta_1=\beta_2=\beta_3=0
$$

This is different from the results we got because we did the hypothesis that $\beta_1=\beta_2=0$ 

This F-test (the one that `.summary` gives) uses the restricted model fo `Y ~ 1` with `q=3`



In [ ]:
#We can get the same results by setting X3=0 as well
print(unrestricted.f_test("X1=0,X2=0,X3=0"))

<F test: F=0.09464540275610382, p=0.9602211077431334, df_denom=6, df_num=3>


**6.**

To get the critical values for a given `α`

`stats.f.ppf(1-α, dfn=, dfd=)`

This will give the value which covers 1-α$^{th}$ of area, i.e. if α=0.05 this give the x at which the area to the left is 95%

### F-test is always a right tailed test as F-statistic is always nonnegative

The denominator is always positive and the denominator is a square. q and df can also never be negative

So we never actually do 1-α/2 as we did for 2 tailed t-test

$$
H_A: \text{at least one } \beta_j ≠ 0
$$
always

**7.** 

For p value, we just need to find the area to the right of our F-statistic

`1 - stats.f.cdf(F, dfn=q, dfd=df)`

Again we do not need to multiply our pvalue with 2 for 2 tailed test


# Exercise

# 1

In [65]:
rdt= pyreadr.read_r(r"C:\Users\khosl\OneDrive\Documents\Py notebook\Books\Real Econometrics\RE2e_Exercise Data_CH05\RE2e_R Data sets_CH05\Ch5_Exercise1_Height_and_Wages_US.RData")

HiWeDF= rdt["dta"]
HiWeDF.head()

,male,white,black,hispanic,mompro2,poppro2,siblings,norest96,norcen96,south96,...,height81,height85,esteem80,athlets,wage96,hgc96,daded79,momed79,age,clubnum
0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,NaN,NaN,NaN,...,65.0,NaN,NaN,NaN,NaN,NaN,8.0,8.0,NaN,NaN
1,0.0,1.0,0.0,0.0,0.0,0.0,8.0,1.0,0.0,0.0,...,62.0,62.0,20.0,0.0,NaN,12.0,8.0,5.0,37.0,2.0
2,0.0,1.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,...,NaN,70.0,24.0,0.0,NaN,12.0,12.0,10.0,34.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,...,67.0,67.0,NaN,0.0,14.632107,14.0,12.0,11.0,33.0,0.0
4,1.0,1.0,0.0,0.0,0.0,0.0,1.0,NaN,NaN,NaN,...,63.0,NaN,23.0,0.0,NaN,NaN,12.0,12.0,NaN,2.0


### a

In [70]:
HiWeOLS1= smf.ols(
    formula="wage96 ~ height85",
    data=HiWeDF
).fit()

HiWeOLS1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 wage96   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     14.59
Date:                Fri, 10 Jul 2026   Prob (F-statistic):           0.000135
Time:                        23:52:09   Log-Likelihood:                -31816.
No. Observations:                6713   AIC:                         6.364e+04
Df Residuals:                    6711   BIC:                         6.365e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -6.9768      5.559     -1.255      0.210     -17.874       3.921
height85       0.3148      0.082      3.820      0.000       0.153       0.476
==============================================================================
Omnibus:                    18019.643   Durbin-Watson:                   1.953
Prob(Omnibus):                  0.000   Jarque-Bera (JB):        653910984.720
Skew:                          32.277   Prob(JB):                         0.00
Kurtosis:                    1530.634   Cond. No.                     1.11e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.11e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [69]:
HiWeOLS2= smf.ols(
    formula="wage96 ~ height85 + height81",
    data=HiWeDF
).fit()

HiWeOLS2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 wage96   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     8.873
Date:                Fri, 10 Jul 2026   Prob (F-statistic):           0.000142
Time:                        23:52:04   Log-Likelihood:                -31302.
No. Observations:                6594   AIC:                         6.261e+04
Df Residuals:                    6591   BIC:                         6.263e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -9.2483      5.775     -1.601      0.109     -20.569       2.072
height85      -0.1068      0.243     -0.440      0.660      -0.582       0.369
height81       0.4575      0.246      1.860      0.063      -0.025       0.940
==============================================================================
Omnibus:                    17663.286   Durbin-Watson:                   1.953
Prob(Omnibus):                  0.000   Jarque-Bera (JB):        625774255.978
Skew:                          32.099   Prob(JB):                         0.00
Kurtosis:                    1510.810   Cond. No.                     1.60e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.6e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [72]:
print(HiWeOLS1.scale)
print(HiWeOLS2.scale)

766.0479302237575
778.0426243651723


Adult hight coefficient changed from 0.3148 to -0.1068 and from significant to insignificant and the variance of regression has also increased


